In [9]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
import joblib

In [18]:
#on entraine le model sur toute l'historique (sans train test split)
df_hist = pd.read_csv(r"C:\Users\Pc\Documents\pfa_stage_project\data\processed\model_data\final_dataset.csv",parse_dates=["Date"])

df_hist = df_hist.set_index("Date")
df_hist = df_hist.asfreq("MS")

df_hist = df_hist.dropna(subset=["KCL", "DAP", "Brent"])

print("Première date :", df_hist.index.min())
print("Dernière date :", df_hist.index.max())
print("Nombre de lignes :", len(df_hist))

Première date : 1967-01-01 00:00:00
Dernière date : 2025-12-01 00:00:00
Nombre de lignes : 707


In [19]:
X_hist = df_hist[["DAP", "Brent"]]
y_hist = df_hist["KCL"]

In [20]:
#reentrainer le model final sur toute l'historique
order = (1, 0, 1)
seasonal_order = (1, 0, 2, 12)

final_model = SARIMAX(
    y_hist,
    exog=X_hist,
    order=order,
    seasonal_order=seasonal_order,
    enforce_invertibility=False,
    enforce_stationarity=False
)

final_model_fitted = final_model.fit(disp=False)

print(final_model_fitted.summary())

c:\Users\Pc\Downloads\anaconda\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\Pc\Downloads\anaconda\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


                                        SARIMAX Results                                        
Dep. Variable:                                     KCL   No. Observations:                  707
Model:             SARIMAX(1, 0, 1)x(1, 0, [1, 2], 12)   Log Likelihood               -3112.492
Date:                                 Mon, 17 Aug 2026   AIC                           6240.984
Time:                                         10:50:12   BIC                           6277.173
Sample:                                              0   HQIC                          6254.991
                                                 - 707                                         
Covariance Type:                                   opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
DAP            0.2140      0.020     10.712      0.000       0.175       0

In [21]:
#charger les nouvelles donnees
df_new = pd.read_csv(r"C:\Users\Pc\Documents\pfa_stage_project\data\processed\model_data\future_exog.csv")

df_new["date"] = pd.to_datetime(df_new["date"])
df_new = df_new.set_index("date")

print(df_new)

            brent_usd_bbl  dap_usd_mt                                  source
date                                                                         
2025-12-01           62.7       627.5  World Bank Pink Sheet (Jan 2026 issue)
2026-01-01           66.8       619.2  World Bank Pink Sheet (Apr 2026 issue)
2026-02-01           71.1       626.5  World Bank Pink Sheet (Apr 2026 issue)
2026-03-01          103.7       658.3  World Bank Pink Sheet (Apr 2026 issue)
2026-04-01          120.4       725.3  World Bank Pink Sheet (Jul 2026 issue)
2026-05-01          107.5       769.5  World Bank Pink Sheet (Jul 2026 issue)
2026-06-01           85.4       783.8  World Bank Pink Sheet (Jul 2026 issue)


In [22]:
df_new=df_new.dropna()

In [24]:
X_new = pd.DataFrame({
    "DAP": df_new["dap_usd_mt"],
    "Brent": df_new["brent_usd_bbl"]
}, index=df_new.index)

print(X_new)

              DAP  Brent
date                    
2025-12-01  627.5   62.7
2026-01-01  619.2   66.8
2026-02-01  626.5   71.1
2026-03-01  658.3  103.7
2026-04-01  725.3  120.4
2026-05-01  769.5  107.5
2026-06-01  783.8   85.4


In [25]:
#prediction
forecast = final_model_fitted.get_forecast(
    steps=len(X_new),
    exog=X_new
)

y_pred = forecast.predicted_mean
conf_int = forecast.conf_int()

c:\Users\Pc\Downloads\anaconda\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
c:\Users\Pc\Downloads\anaconda\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(


In [26]:
resultats = pd.DataFrame({
    "Date": X_new.index,
    "DAP": X_new["DAP"].values,
    "Brent": X_new["Brent"].values,
    "KCL_Predicted": y_pred.values
})

print(resultats)

        Date    DAP  Brent  KCL_Predicted
0 2025-12-01  627.5   62.7     362.453260
1 2026-01-01  619.2   66.8     358.185683
2 2026-02-01  626.5   71.1     356.530227
3 2026-03-01  658.3  103.7     373.394753
4 2026-04-01  725.3  120.4     391.926474
5 2026-05-01  769.5  107.5     395.213402
6 2026-06-01  783.8   85.4     387.094220


In [27]:
#sauvegarde
#fichier final de prediction
predictions_df = pd.DataFrame({
    "Date": X_new.index,
    "DAP": X_new["DAP"].values,
    "Brent": X_new["Brent"].values,
    "KCL_Predicted": y_pred.values
})
prediction_path=prediction_path = r"C:\Users\Pc\Documents\pfa_stage_project\data\processed\model_data\predictions_KCL_2026.csv"

In [28]:
predictions_df.to_csv(prediction_path,index=False)
print("Prévisions sauvegardées avec succès.")
print(prediction_path)

Prévisions sauvegardées avec succès.
C:\Users\Pc\Documents\pfa_stage_project\data\processed\model_data\predictions_KCL_2026.csv


In [29]:
print(predictions_df)

        Date    DAP  Brent  KCL_Predicted
0 2025-12-01  627.5   62.7     362.453260
1 2026-01-01  619.2   66.8     358.185683
2 2026-02-01  626.5   71.1     356.530227
3 2026-03-01  658.3  103.7     373.394753
4 2026-04-01  725.3  120.4     391.926474
5 2026-05-01  769.5  107.5     395.213402
6 2026-06-01  783.8   85.4     387.094220
